# Lógica Booleana e Simplificação Algébrica

**Capítulo aplicado:** 2 (Lógica Booleana).

Implementa em Python puro:
- operadores **AND, OR, NOT** da álgebra de Boole;
- geração automática de **tabela-verdade**;
- extração de expressão por **mintermos** (Soma de Produtos);
- exemplo de **simplificação algébrica** (cap. 3.4.5 do PDF):
  $(A \\cdot B') + (A \\cdot B) = A$.

## 1. Operadores da álgebra de Boole

In [ ]:
def AND(a, b):
    """1 se ambos = 1; senao 0."""
    return 1 if (a == 1 and b == 1) else 0


def OR(a, b):
    """1 se pelo menos um = 1."""
    return 1 if (a == 1 or b == 1) else 0


def NOT(a):
    """Inverte 0/1."""
    return 1 - a

## 2. Tabela-verdade automática

Para `n` variáveis, a tabela tem `2^n` linhas (cap. 2).

In [ ]:
def tabela_verdade_2(funcao):
    linhas = []
    for a in (0, 1):
        for b in (0, 1):
            linhas.append((a, b, funcao(a, b)))
    return linhas


def imprimir_tabela_verdade(linhas, nomes=("A", "B", "X")):
    print(f"  {nomes[0]} | {nomes[1]} | {nomes[2]}")
    print("  --+---+--")
    for linha in linhas:
        print(f"  {linha[0]} | {linha[1]} | {linha[2]}")

## 3. Extração por mintermos (Soma de Produtos)

Regra do capítulo 2.9: para cada linha em que X=1, monta-se um
**produto** (AND). Variável em 1 entra direta; variável em 0 entra
**negada** (apóstrofe). A expressão final é a **soma** (OR) de
todos os mintermos.

In [ ]:
def extrair_mintermos(linhas, nomes=("A", "B")):
    mintermos = []
    for linha in linhas:
        saida = linha[-1]
        entradas = linha[:-1]
        if saida == 1:
            termo = []
            for i, valor in enumerate(entradas):
                if valor == 1:
                    termo.append(nomes[i])
                else:
                    termo.append(nomes[i] + "'")
            mintermos.append("(" + " AND ".join(termo) + ")")
    if not mintermos:
        return "0"
    return " OR ".join(mintermos)

## 4. Simplificação algébrica do capítulo 3.4.5

Exemplo do PDF: extrair por mintermos a expressão da tabela em que
$X=1$ apenas quando $A=1$, e mostrar que ela se reduz a $X = A$.

Passos (leis da álgebra de Boole):
1. $X = (A \\cdot B') + (A \\cdot B)$
2. Fatoração: $X = A \\cdot (B' + B)$
3. Complemento: $B' + B = 1$, então $X = A \\cdot 1$
4. Identidade: $A \\cdot 1 = A$, então $X = A$

In [ ]:
def explicar_simplificacao_exemplo():
    return [
        "X = (A AND NOT B) OR (A AND B)",
        "X = A AND (NOT B OR B)     # fatoracao",
        "X = A AND 1                # complemento: NOT B OR B = 1",
        "X = A                      # identidade:  A AND 1 = A",
    ]


def verificar_simplificacao():
    """Confirma numericamente que (A.B')+(A.B) == A para todos os casos."""
    for a in (0, 1):
        for b in (0, 1):
            original = OR(AND(a, NOT(b)), AND(a, b))
            if original != a:
                return False
    return True

## 5. Aplicação à colônia

Conversão das leituras em variáveis booleanas e combinação com
AND/OR/NOT (cap. 2.4).

In [ ]:
def avaliar_condicoes(energia, consumo, previsao_tempestade):
    return {
        "energia_critica": 1 if energia < 30 else 0,
        "energia_baixa":   1 if energia < 50 else 0,
        "consumo_alto":    1 if consumo >= 60 else 0,
        "tempestade":      1 if previsao_tempestade else 0,
    }


def expressao_modo_emergencia(cond):
    return AND(cond["energia_critica"], cond["consumo_alto"])


def expressao_modo_economia(cond):
    parte1 = AND(cond["tempestade"], cond["energia_baixa"])
    parte2 = AND(AND(cond["energia_baixa"], cond["consumo_alto"]),
                 NOT(cond["energia_critica"]))
    return OR(parte1, parte2)

## 6. Demonstração

In [ ]:
print("Tabela-verdade de X = A OR B:")
linhas = tabela_verdade_2(lambda a, b: OR(a, b))
imprimir_tabela_verdade(linhas)
print("\nExpressao por mintermos:")
print("  X =", extrair_mintermos(linhas))

print("\n" + "-" * 50)
print("Tabela-verdade de X = (A AND NOT B) OR (A AND B):")
linhas2 = tabela_verdade_2(lambda a, b: OR(AND(a, NOT(b)), AND(a, b)))
imprimir_tabela_verdade(linhas2)

print("\nSimplificacao algebrica passo a passo:")
for passo in explicar_simplificacao_exemplo():
    print("  " + passo)
print("\nVerificacao numerica:", verificar_simplificacao())

print("\n" + "-" * 50)
print("Aplicacao a colonia:")
c = avaliar_condicoes(energia=25, consumo=70, previsao_tempestade=False)
print("  Variaveis booleanas:", c)
print("  Modo emergencia (AND) =", expressao_modo_emergencia(c))
print("  Modo economia (OR/AND) =", expressao_modo_economia(c))